<a href="https://colab.research.google.com/github/jeffRnR/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeffRnR/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Task type: Binary Classification

The output I need is a yes/no decision for each page: *is this page declining*
That is binary classification - predicting one of two classes. The ranked queue from ML-02 is built on top of this: first classify which pages are declining, then rank those by confidence.

**Why this fits:** The content team needs a clear action boundary. A page is either worth a human reviewer's time or it isn't. The scoring system from ML-02 is actually a classification model's probability output, sorted into a ranked list. The core ML task underneath is still binary.

**Why the other types don't fit:**

- **Regression:** We don't need a continuous number, The business action is binary - review or don't review. A regression model predicting "decline percentage" would still need a threshold to act on, which just recreates classification.
- **Clustering:** This is unsupervised. But we have a clear business outcome from the date (`trend_direction == "down`), and we need to predict on new pages, not just explore patterns

- **Ranking:** Ranking is what we *do with the output* (sort by probability),
but the core task is first *identifying* which pages belong in the queue. Ranking without classification is sorting a list without knowing what should be on it.

- **Scoring:** A score is just a continous output. The fundamental task is still separating pages into two groups - the score is the model's confidence, not the task itself.

**Business objective:** Maximize the efficiency of content refresh efforts by automatically identifying pages that are declining in performance, so the content team focuses on pages that will benefit most from updates.

**ML objective:** Predict `is_declining` (1 if `trend_direction == "down"`, else 0) for each page, based on observable signals like content age, traffic trends, and engagement metrics.


In [ ]:
# Verify binary classification structure
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Derive the binary target exactly as the pipeline defines it
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("Binary classification target verification:")
print(f"  Dataset: {df.shape[0]:,} pages, {df.shape[1]} columns")
print(f"  Classes: {sorted(df['is_declining'].unique())}")
print(f"  Class distribution:")
print(f"    Not declining (0): {(df['is_declining'] == 0).sum():,} ({(df['is_declining'] == 0).mean():.1%})")
print(f"    Declining (1):     {(df['is_declining'] == 1).sum():,} ({(df['is_declining'] == 1).mean():.1%})")

# Confirm exactly 2 classes
assert df['is_declining'].nunique() == 2, "Expected exactly 2 classes for binary classification"
print("\nConfirmed: This is a binary classification problem.")

Binary classification target verification:
  Dataset: 30,000 pages, 45 columns
  Classes: [np.int64(0), np.int64(1)]
  Class distribution:
    Not declining (0): 13,738 (45.8%)
    Declining (1):     16,262 (54.2%)

Confirmed: This is a binary classification problem.


## 2. Target or proxy

**What would you predict?** Whether a page is declining in performance.

**Where does that label come from?** A defined rule, not an observed outcome.

**Ideal target:** Whether refreshing a page actually improves its performance. This is the true business value - did the refresh work?

**Why the ideal target is unavailable:** We cannot know the future. The "refresh helped" label would require running the refresh, waiting weeks, and measuring results. This is an experimental outcome, not an observed one in this dataset.

**Proxy target:** `is_declining = (trend_direction == "down")`

**Why this proxy was selected:**
- It is **measurable** from historical data - no experiment needed
- It is **actionable** - a page with a downward trend is a logical candidate for refresh
- It is **correlated with business value** - pages in decline are the ones most likely to benefit from content updates

**Assumptions:**
1. A downward trend means the page needs refresh. *Reasonable:* Content freshness is a known ranking factor
2. Refreshing a declining page will reverse the trend. *Reasonable:* The business runs a refresh program because it has evidence this works
3. `trend_direction` accurately captures performance direction. *Reasonable:* It is computed from multiple metrics over a meaningful window.

**Limitations:**
- Seasonal pages (holiday content) decline off-season without needing refresh
- External events (competitor launches) cause decline that refresh cannot fix
- The proxy cannot distinguish "declining and fixable" from "declining because topic is dead"
- We may miss pages that are stable now but about to decline

**Alternative considered:** Using `trend_pct` (continuous) as a regression target. Rejected because the business action is still binary (review or don't review), and `trend_pct` is label leakage - it directly reveals how much the page declined, which would not be available at prediction time.

**Recommendation:** The `is_declining` proxy is the strongest available choice. It is directly actionable, computable from existng data, and aligns with the current business process.


In [ ]:
# Demonstrate the proxy target and its relationship to the ideal target

print("IDEAL TARGET (unavailable):")
print("  'Did refreshing this page improve its traffic?'")
print("  Requires: run refresh → wait 4-8 weeks → measure outcome")
print("  Status: Experimental, not observed in this dataset.\n")

print("PROXY TARGET (available):")
print("  is_declining = (trend_direction == 'down')")
print("  Derived from: trend_direction column (observed historical data)\n")

print("Proxy derivation verification:")
print(df['trend_direction'].value_counts().to_string())
print()
print("Mapped to binary:")
print(df['is_declining'].value_counts().to_string())

# Critical: show that trend_direction and trend_pct are LABEL SOURCES
print(f"\n{'='*60}")
print("LEAKAGE WARNING")
print(f"{'='*60}")
print("These columns CREATE the label — they must NEVER be model features:")
print(f"  trend_direction: {df['trend_direction'].unique()}")
print(f"  trend_pct range: {df['trend_pct'].min():.3f} to {df['trend_pct'].max():.3f}")
print("\nUsing these as features would be data leakage.")

# Show examples of each class — use columns that actually exist
print(f"\n{'='*60}")
print("EXAMPLE: Not declining (is_declining = 0)")
print(f"{'='*60}")
example_cols = ['content_id', 'trend_direction', 'trend_pct', 'content_age_days', 'impressions_90d']
print(df[df['is_declining'] == 0][example_cols].head(2).to_string())

print(f"\n{'='*60}")
print("EXAMPLE: Declining (is_declining = 1)")
print(f"{'='*60}")
print(df[df['is_declining'] == 1][example_cols].head(2).to_string())

IDEAL TARGET (unavailable):
  'Did refreshing this page improve its traffic?'
  Requires: run refresh → wait 4-8 weeks → measure outcome
  Status: Experimental, not observed in this dataset.

PROXY TARGET (available):
  is_declining = (trend_direction == 'down')
  Derived from: trend_direction column (observed historical data)

Proxy derivation verification:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Mapped to binary:
is_declining
1    16262
0    13738

LEAKAGE WARNING
These columns CREATE the label — they must NEVER be model features:
  trend_direction: ['down' 'stable' 'new' 'up' 'flat']
  trend_pct range: -100.000 to 44900.000

Using these as features would be data leakage.

EXAMPLE: Not declining (is_declining = 0)
             content_id trend_direction  trend_pct  content_age_days  impressions_90d
3  content_331d6c4de07b          stable      -13.8               463            11751
7  content_a63219c6e95a          stable      

## 3. Success metric
**One metric I can defend: Precision@50**

Precision@K measures what fraction of the top-K predicted declining pages are actually declining. If we recommend 50 pages to review, Precision@50 tells us how much of those 50 were truly declining.

**Why Precision@50 matches the business objective:** The content team has limited capacity. They can only review a certain number of pages per week. We need the pages we *do* recommend to be genuinely worth their time. Precision@K directly measures the quality of the queue.

**What number means 'good':** The baseline hand-rule achieves Precision@50 = 0.24. A successful ML model should achieve **Precision@50 >= 0.70** - approximately a **3x lift**. This means 7 of 10 recommended pages are genuinely declining.

**Secondary metric: ROC AUC**

ROC AUC measures the model's ability to separate declining from stable pages across all thresholds. It tells us the model's overall discriminative power.

**What "good" looks like:** ROC AUC ≥ 0.80 indicates strong separation.

**Why other metrics are less appropriate:**

- **Accuracy:** Misleading with imbalanced classes. If 65% of pages are stable, predicting "not declining" for everything gives 65% accuracy but is useless.
- **F1 Score:** Balances precision and recall, but our need is asymmetric - precision matters more because false positives waste real editor time.
- **Recall:** Important but secondary. Missing a declining page this cycle means we catch it next cycle.Recommending a wrong page wastes effort that .cannot be recovered.
- **RMSE / MAE:** Regression metrics. We are doing classification.
- **NDCG / MAP:** Ranking metrics. Useful for ordering the queue, but the core task is classification

**Success criteria:**

| Metric | Threshold | Meaning |
|---|---|---|
| Precision@50 | >= 0.70 | 7 of 10 top recommendations are genuinely declining |
| ROC AUC | >= 0.80 | Strong model discrimination |
| Recall | >= 0.60 | We catch most declining pages (secondary) |

In [ ]:
# Demonstrate why Precision@50 is the right metric

# Simulate baseline rule: content_age > 365
baseline_preds = (df['content_age_days'] > 365).astype(int)
baseline_hits = df[baseline_preds == 1]['is_declining'].sum()
baseline_total = baseline_preds.sum()
baseline_precision = baseline_hits / min(50, baseline_total) if baseline_total > 0 else 0

print(f"Baseline rule (content_age > 365) precision@50: ~{baseline_precision:.3f}")
print(f"  (Pipeline reports actual baseline Precision@50: 0.240)")
print(f"  Meaning: ~12 of every 50 pages flagged by the rule are actually declining\n")

# Show class imbalance — why accuracy is misleading
class_dist = df['is_declining'].value_counts(normalize=True)
print("Class distribution (why accuracy fails):")
print(f"  Not declining (0): {class_dist[0]:.1%}")
print(f"  Declining (1):     {class_dist[1]:.1%}")
print(f"\nAlways predicting 'not declining' gives accuracy: {class_dist[0]:.1%}")
print("This is useless — we never recommend any refreshes!\n")

# Business cost analysis
print("Business cost of errors:")
print("  False Positive (flag healthy page): Editor wastes 2-4 hours")
print("  False Negative (miss declining page): Caught next cycle, page keeps declining")
print("  → False positives are more expensive per-instance → optimize PRECISION\n")

# ML target from pipeline
print(f"{'='*60}")
print("ML TARGET (from pipeline documentation)")
print(f"{'='*60}")
print("Random Forest Precision@50: 0.740 (~37 of top 50 correct)")
print("Baseline Precision@50:      0.240 (~12 of top 50 correct)")
print("Improvement: 3.1x")
print("Source: outputs/model_results.json from reference pipeline")

Baseline rule (content_age > 365) precision@50: ~54.220
  (Pipeline reports actual baseline Precision@50: 0.240)
  Meaning: ~12 of every 50 pages flagged by the rule are actually declining

Class distribution (why accuracy fails):
  Not declining (0): 45.8%
  Declining (1):     54.2%

Always predicting 'not declining' gives accuracy: 45.8%
This is useless — we never recommend any refreshes!

Business cost of errors:
  False Positive (flag healthy page): Editor wastes 2-4 hours
  False Negative (miss declining page): Caught next cycle, page keeps declining
  → False positives are more expensive per-instance → optimize PRECISION

ML TARGET (from pipeline documentation)
Random Forest Precision@50: 0.740 (~37 of top 50 correct)
Baseline Precision@50:      0.240 (~12 of top 50 correct)
Improvement: 3.1x
Source: outputs/model_results.json from reference pipeline


## 4. The unit of analysis, as a real dataframe

**One row = one web page (content item)**

Each row represents a single content item's current state and 90-day historical performance. The refresh action happens at the content level — editors open a specific piece of content, read the reason code, and decide whether to rewrite, expand, protect, or prune.

**Why this granularity is correct:**
- The business action (refresh) is content-level
- Features like `content_age_days`, `days_since_last_update`, and `engagement_rate` are naturally content-level attributes
- The client-holdout split (used in the reference pipeline) groups by `client_id` to prevent leakage — no client's content appears in both train and test

**Key columns:**

| Column | Meaning | Role |
|---|---|---|
| `content_id` | Unique content identifier | Identifier (not a feature) |
| `client_id` | Site/client this content belongs to | Grouping for train/test split |
| `content_type` | Type of content (e.g., keyword article) | Feature |
| `main_intent` | Search intent (transactional, informational) | Feature |
| `word_count` | Number of words in content | Feature |
| `char_count` | Number of characters | Feature |
| `search_volume` | Keyword search volume | Feature |
| `competition` / `competition_level` | Keyword competition metrics | Feature |
| `cpc` | Cost per click for keyword | Feature |
| `provider_used` / `model_used` | AI provider/model used to generate content | Feature |
| `impressions_90d` | Total impressions in last 90 days | Feature |
| `clicks_90d` | Total clicks in last 90 days | Feature |
| `pageviews_90d` | Total pageviews in last 90 days | Feature |
| `sessions_90d` | Total sessions in last 90 days | Feature |
| `users_90d` | Unique users in last 90 days | Feature |
| `engaged_sessions_90d` | Sessions with engagement in last 90 days | Feature |
| `ai_sessions_90d` | AI-referred sessions in last 90 days | Feature |
| `scroll_events_90d` | Scroll events in last 90 days | Feature |
| `days_with_impressions` | Days with any impressions in 90d window | Feature |
| `days_with_sessions` | Days with any sessions in 90d window | Feature |
| `impressions_last_30d` | Impressions in last 30 days | Feature |
| `clicks_last_30d` | Clicks in last 30 days | Feature |
| `sessions_last_30d` | Sessions in last 30 days | Feature |
| `impressions_prev_30d` | Impressions in previous 30 days | Feature |
| `clicks_prev_30d` | Clicks in previous 30 days | Feature |
| `sessions_prev_30d` | Sessions in previous 30 days | Feature |
| `content_age_days` | How old the content is (days) | Feature |
| `age_tier` / `age_tier_order` | Binned age categories | Feature (or derived) |
| `days_since_last_update` | Days since content was last updated | Feature |
| `freshness_tier` | Content freshness category | Feature (or derived) |
| `word_count_tier` / `char_count_tier` | Binned size categories | Feature (or derived) |
| `ctr` | Click-through rate | Feature |
| `avg_position` | Average search position | Feature |
| `engagement_rate` | Engagement rate | Feature |
| `scroll_rate` | Scroll rate | Feature |
| `ai_traffic_pct` | Percentage of traffic from AI sources | Feature |
| `impression_tier` | Impression volume category | Feature (or derived) |
| `position_tier` | Position category | Feature (or derived) |
| `trend_direction` | 'up', 'down', 'stable', 'new', 'flat' | **LABEL SOURCE — never a feature** |
| `trend_pct` | Numeric trend percentage | **LABEL SOURCE — never a feature** |

In [ ]:
# Load and display the unit of analysis

print(f"{'='*70}")
print("UNIT OF ANALYSIS: ONE ROW = ONE WEB PAGE")
print(f"{'='*70}")

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Each row is one page with its performance metrics.")

print(f"\n{'='*70}")
print("COLUMN NAMES")
print(f"{'='*70}")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n{'='*70}")
print("DATA TYPES")
print(f"{'='*70}")
print(df.dtypes.to_string())

print(f"\n{'='*70}")
print("FIRST 5 ROWS")
print(f"{'='*70}")
print(df.head().to_string())

print(f"\n{'='*70}")
print("SUMMARY STATISTICS (numeric columns)")
print(f"{'='*70}")
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(df[numeric_cols].describe().round(2).transpose().to_string())

print(f"\n{'='*70}")
print("MISSING VALUES")
print(f"{'='*70}")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_count', ascending=False)
if len(missing_df) > 0:
    print(missing_df.to_string())
else:
    print("No missing values detected.")

UNIT OF ANALYSIS: ONE ROW = ONE WEB PAGE

Shape: 30,000 rows × 45 columns
Each row is one page with its performance metrics.

COLUMN NAMES
   1. content_id
   2. client_id
   3. search_volume
   4. competition
   5. competition_level
   6. cpc
   7. content_type
   8. main_intent
   9. word_count
  10. char_count
  11. provider_used
  12. model_used
  13. impressions_90d
  14. clicks_90d
  15. pageviews_90d
  16. sessions_90d
  17. users_90d
  18. engaged_sessions_90d
  19. ai_sessions_90d
  20. scroll_events_90d
  21. days_with_impressions
  22. days_with_sessions
  23. impressions_last_30d
  24. clicks_last_30d
  25. sessions_last_30d
  26. impressions_prev_30d
  27. clicks_prev_30d
  28. sessions_prev_30d
  29. content_age_days
  30. age_tier
  31. age_tier_order
  32. days_since_last_update
  33. freshness_tier
  34. word_count_tier
  35. char_count_tier
  36. ctr
  37. avg_position
  38. engagement_rate
  39. scroll_rate
  40. ai_traffic_pct
  41. impression_tier
  42. position_ti

## 5. Why ML beats a fixed rule here

### What makes this pattern too messy for an if-statement

The content refresh problem has five characteristics that break fixed rules:

**1. Multiple signals that interact**

A page's decline risk depends on `content_age_days`, `engagement_rate`, `trend_pct`, `impressions_90d`, `days_since_last_update`, `content_type`, and more — all at once. A fixed rule can check one or two thresholds, but it cannot learn that `content_age_days` only matters when `engagement_rate` is already low, or that `trend_pct` is the dominant signal for `keyword article` but not for `landing page`. ML models discover these interactions from data.

**2. Different rules for different content types**

`content_type` has multiple categories (keyword article, landing page, etc.). The "stale" threshold for a news article is 30 days. For a reference guide, it might be 2 years. A single if-statement cannot hold multiple thresholds without becoming unreadable. ML learns a separate decision boundary per content type automatically.

**3. The signals change meaning over time**

Search algorithms shift. What made a page "healthy" in January might not apply in June. A fixed rule written in January is wrong by June. ML retraining absorbs these shifts — the model learns from recent data what "declining" looks like *now*, not what it looked like six months ago.

**4. Thresholds are continuous, not binary**

A page with 364 days since update and one with 366 days are essentially identical. A fixed rule treats them as completely different (review vs. skip). ML uses the full continuous value and learns a smooth risk curve, not a cliff at day 365.

**5. The cost of errors is asymmetric**

A false positive (flagging a healthy page) wastes 2-4 hours of editor time. A false negative (missing a declining page) means lost traffic that may never recover. A fixed rule cannot optimize for this trade-off. ML can — by tuning the decision threshold on the validation set to minimize the costlier error.

### Why this specifically breaks a fixed rule

A fixed rule for this dataset would need to look like:

```python
if (content_age_days &gt; 365 and content_type == "keyword article"
    and engagement_rate &lt; 0.05 and trend_pct &lt; -20):
    return "review"
elif (content_age_days &gt; 730 and content_type == "landing page"
      and impressions_90d &gt; 1000):
    return "review"
# ... dozens more branches
else:
    return "skip"

In [ ]:
# Demonstrate why fixed rules fail on real data

# Example 1: Evergreen exception — old but stable
old_but_stable = df[(df['content_age_days'] > 365) & (df['is_declining'] == 0)]
print(f"EXAMPLE 1: EVERGREEN EXCEPTION")
print(f"Pages older than 365 days that are NOT declining: {len(old_but_stable):,}")
print(f"A fixed rule ('review if > 365 days') would waste effort on all of these.")
cols = ['content_id', 'content_age_days', 'impressions_90d', 'engagement_rate', 'is_declining']
print(old_but_stable[cols].head(3).to_string())
print()

# Example 2: False stable trap — recent but declining
recent_but_declining = df[(df['content_age_days'] < 180) & (df['is_declining'] == 1)]
print(f"EXAMPLE 2: FALSE STABLE TRAP")
print(f"Pages newer than 180 days that ARE declining: {len(recent_but_declining):,}")
print(f"A fixed rule ('skip if < 180 days') would miss all of these.")
print(recent_but_declining[cols].head(3).to_string())
print()

# Example 3: Rule precision vs. ML target
rule_preds = (df['content_age_days'] > 365).astype(int)
rule_precision = df[rule_preds == 1]['is_declining'].mean()
print(f"EXAMPLE 3: RULE PRECISION")
print(f"Fixed rule (content_age > 365) precision: {rule_precision:.3f}")
print(f"Meaning: only {rule_precision:.1%} of pages flagged by the rule are actually declining")
print(f"ML target (from pipeline): Precision@50 = 0.740")
print(f"ML would be {0.740/rule_precision:.1f}x more precise than this simple rule")
print()

# Show that optimal threshold varies by page characteristics
print(f"{'='*60}")
print("THRESHOLD VARIES BY CONTENT TYPE")
print(f"{'='*60}")
for ctype, group in df.groupby('content_type'):
    if len(group) > 100:  # Only show types with enough data
        declining_age = group[group['is_declining'] == 1]['content_age_days'].median()
        stable_age = group[group['is_declining'] == 0]['content_age_days'].median()
        print(f"{ctype}: median age of declining = {declining_age:.0f} days, stable = {stable_age:.0f} days")
print("\nA single fixed threshold cannot fit all content types — ML learns type-specific patterns.")

EXAMPLE 1: EVERGREEN EXCEPTION
Pages older than 365 days that are NOT declining: 3,649
A fixed rule ('review if > 365 days') would waste effort on all of these.
              content_id  content_age_days  impressions_90d  engagement_rate  is_declining
3   content_331d6c4de07b               463            11751             1.28             0
7   content_a63219c6e95a               445             1724             3.57             0
30  content_249298388b45               438              176             0.00             0

EXAMPLE 2: FALSE STABLE TRAP
Pages newer than 180 days that ARE declining: 7,523
A fixed rule ('skip if < 180 days') would miss all of these.
             content_id  content_age_days  impressions_90d  engagement_rate  is_declining
2  content_9aa793d4d895               141            12581              0.0             1
5  content_d4084a4bc775               147             3970              0.0             1
6  content_9a34b442b552                90               20    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.